# Bundling OpenMM simulations for NanoVer

This notebook demonstrates how to take an OpenMM molecular simulation setup and bundle it into a compact format that can be used in NanoVer for convenience.

## OpenMM simulation setup

First we set up an OpenMM simulation (example adapted from [OpenMM documentation](https://docs.openmm.org/latest/userguide/application/02_running_sims.html)).

In [1]:
from openmm import unit, LangevinMiddleIntegrator
from openmm.app import ForceField, PME, HBonds, PDBFile, Simulation

pdb = PDBFile("../systems/17-ala.pdb")
forcefield = ForceField('amber19-all.xml', 'amber19/tip3pfb.xml')

system = forcefield.createSystem(
    pdb.topology,
    nonbondedMethod=PME,
    nonbondedCutoff=1*unit.nanometer,
    constraints=HBonds,
    removeCMMotion=False,
)

integrator = LangevinMiddleIntegrator(
    300*unit.kelvin,
    1/unit.picosecond,
    0.002*unit.picoseconds,
)

simulation = Simulation(pdb.topology, system, integrator)
simulation.context.setPositions(pdb.positions)

In [2]:
# NBVAL_SKIP
simulation.minimizeEnergy()

## Bundling the simulation

Next we bundle the constructed simulation into a single file; a compressed archive of OpenMM's own serialization of the simulation components.

In [3]:
from nanover.openmm import bundle_openmm_simulation

bundle_openmm_simulation(simulation, outfile="17-ala.openmm.zip")

## Unbundling the simulation

Unbundling is equally simple:

In [4]:
from nanover.openmm import unbundle_openmm_simulation

simulation_copy = unbundle_openmm_simulation("17-ala.openmm.zip")

In [5]:
print(simulation.topology, "\n", simulation_copy.topology)

<Topology; 1 chains, 17 residues, 173 atoms, 172 bonds> 
 <Topology; 1 chains, 17 residues, 173 atoms, 172 bonds>


## NanoVer simulation from OpenMM bundle

Additionally one can create a NanoVer ready simulation directly from a bundle and serve it over the network as usual.

In [6]:
from nanover.openmm import OpenMMSimulation

omm_sim = OpenMMSimulation.from_bundle_path("17-ala.openmm.zip")
omm_sim.load()

In [7]:
from nanover.app import OmniRunner

imd_runner = OmniRunner.with_basic_server(omm_sim, port=0, name="openmm example")
imd_runner.print_basic_info()
imd_runner.load(0)

Serving "openmm example" (ws://localhost:59893), discoverable on all interfaces on port 54545
Available simulations:
[0]: "17-ala.openmm"
Switched to [0]: "17-ala.openmm"
Switched to [0]: "17-ala.openmm"
